# PPE Detection — YOLO26m Training
**Model:** YOLO26m | **Classes:** helmet, vest, person, no-helmet, no-vest | **Hardware:** Kaggle Dual GPU

### How to use
1. Upload your  folder as a Kaggle dataset
2. Add it to this notebook via *Add Data*
3. Set  in Cell 2 to match your Kaggle dataset slug
4. Run all cells

In [3]:
# ============================================================
# 1. INSTALL
# ============================================================
!pip install -U ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.2 MB/s eta 0:00:0000:01


In [4]:
import os, shutil, yaml, glob, torch
from pathlib import Path
from ultralytics import YOLO

print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
print(f"GPU count:  {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory/1e9:.1f} GB")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch:    2.10.0+cu128
CUDA:       True
GPU count:  2
  GPU 0: Tesla T4 | 15.6 GB
  GPU 1: Tesla T4 | 15.6 GB


## Step 1 — Configure paths

Set  to the slug of the Kaggle dataset you uploaded.
For example, if your dataset URL is , set it to .

In [10]:
# ============================================================
# 2. PATHS
# ============================================================
INPUT_DIR   = "/kaggle/input/datasets/smshezanahmed/merged-construction-ppe-dataset/merged_dataset"
WORKING_DIR = "/kaggle/working/ppe_data"

if not os.path.exists(INPUT_DIR):
    raise FileNotFoundError("Dataset not found at: " + INPUT_DIR)

print("Input dataset found:", INPUT_DIR)
print("Contents:", os.listdir(INPUT_DIR))

Input dataset found: /kaggle/input/datasets/smshezanahmed/merged-construction-ppe-dataset/merged_dataset
Contents: ['data.yaml', 'valid', 'test', 'train']


## Step 2 — Copy dataset to writable directory

In [11]:
# ============================================================
# 3. COPY DATASET TO WRITABLE FOLDER
# ============================================================
print("Copying dataset to writable directory...")

if os.path.exists(WORKING_DIR):
    shutil.rmtree(WORKING_DIR)

shutil.copytree(INPUT_DIR, WORKING_DIR)
print(f"Copied to: {WORKING_DIR}")

# Show folder structure
for split in ["train", "val", "valid", "test"]:
    img_dir = Path(WORKING_DIR) / split / "images"
    lbl_dir = Path(WORKING_DIR) / split / "labels"
    if img_dir.exists():
        n_imgs = len(list(img_dir.glob("*")))
        n_lbls = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
        print(f"  {split:6}: {n_imgs:,} images | {n_lbls:,} labels")

Copying dataset to writable directory...
Copied to: /kaggle/working/ppe_data
  train : 25,178 images | 25,188 labels
  valid : 2,461 images | 2,461 labels
  test  : 1,414 images | 1,414 labels


## Step 3 — Fix and validate data.yaml

In [7]:
# ============================================================
# 4. FIX YAML PATHS
# ============================================================
yaml_files = glob.glob(os.path.join(WORKING_DIR, "*.yaml"))
if not yaml_files:
    yaml_files = glob.glob(os.path.join(WORKING_DIR, "**/*.yaml"), recursive=True)
if not yaml_files:
    raise FileNotFoundError("No .yaml file found in dataset!")

yaml_file = yaml_files[0]
print(f"Found config: {yaml_file}")

with open(yaml_file) as f:
    data = yaml.safe_load(f)

# Fix absolute paths so YOLO can find images on Kaggle
data["path"] = WORKING_DIR
data["train"] = "train/images"
data["test"]  = "test/images"

# Handle both "val" and "valid" folder names
if os.path.exists(os.path.join(WORKING_DIR, "valid", "images")):
    data["val"] = "valid/images"
else:
    data["val"] = "val/images"

# Remove download key if present (not needed on Kaggle)
data.pop("download", None)

with open(yaml_file, "w") as f:
    yaml.dump(data, f, default_flow_style=False)

CLASS_NAMES = data["names"] if isinstance(data["names"], list) else list(data["names"].values())
print(f"Classes ({data['nc']}): {CLASS_NAMES}")
print(f"train: {data['train']}")
print(f"val:   {data['val']}")
print(f"test:  {data['test']}")

Found config: /kaggle/working/ppe_data/data.yaml
Classes (5): ['helmet', 'vest', 'person', 'no-helmet', 'no-vest']
train: train/images
val:   valid/images
test:  test/images


## Step 4 — Dataset sanity check

In [8]:
# ============================================================
# 5. SANITY CHECK — class balance
# ============================================================
from collections import defaultdict

counts = defaultdict(int)
root = Path(WORKING_DIR)

for split in ["train", "val", "valid", "test"]:
    lbl_dir = root / split / "labels"
    if not lbl_dir.exists():
        continue
    for lbl in lbl_dir.glob("*.txt"):
        for line in lbl.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1

total = sum(counts.values())
print(f"Total annotations: {total:,}")
print(f"  {'Class':<15} {'Annotations':>12} {'%':>8}")
print("-" * 38)
for i, name in enumerate(CLASS_NAMES):
    n = counts.get(i, 0)
    print(f"  {name:<15} {n:>12,} {n/total*100:>7.1f}%")

Total annotations: 126,057
  Class            Annotations        %
--------------------------------------
  helmet                33,869    26.9%
  vest                  32,048    25.4%
  person                33,522    26.6%
  no-helmet             11,700     9.3%
  no-vest               14,918    11.8%


## Step 5 — Train YOLO26m

In [9]:
# ============================================================
# 6. TRAIN YOLO26m — DUAL GPU
# ============================================================
model = YOLO("yolo26m.pt")   # Downloads pretrained weights automatically

results = model.train(
    data=yaml_file,
    epochs=60,
    imgsz=640,
    batch=48,              # 32 total (16 per GPU)
    device=[0, 1],         # Dual GPU
    optimizer="auto",      # Auto-selects MuSGD for long runs
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=12,           # Early stopping
    mosaic=1.0,
    mixup=0.1,
    close_mosaic=10,       # Disable mosaic in final 10 epochs
    save=True,
    save_period=10,        # Checkpoint every 10 epochs
    project="/kaggle/working/runs",
    name="yolo26m_ppe",
    exist_ok=True,
    plots=True,
    val=True,
    cache=True,
    workers=4,
)

print("Training complete!")
print(f"Best mAP@50:    {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"Best mAP@50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=48, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ppe_data/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26m_ppe, nbs=64, nms=Fal

AttributeError: 'NoneType' object has no attribute 'results_dict'

## Step 6 — Validate and per-class results

In [20]:
# ============================================================
# 7. VALIDATE BEST MODEL — per-class AP
# ============================================================
import pandas as pd

best_pt = "/kaggle/input/models/smshezanahmed/yolo26-ppe-42mb/pytorch/default/1/best (3).pt"
model_best = YOLO(best_pt)
metrics = model_best.val(data=yaml_file, imgsz=640, device=0)

rows = []
if hasattr(metrics.box, "ap50") and hasattr(metrics.box, "ap_class_index"):
    for cls_idx, ap50, ap in zip(
        metrics.box.ap_class_index,
        metrics.box.ap50,
        metrics.box.ap
    ):
        name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else str(cls_idx)
        rows.append({"Class": name, "AP@50": round(float(ap50), 4), "AP@50-95": round(float(ap), 4)})

df = pd.DataFrame(rows)
print(df.to_markdown(index=False))
print(f"Overall mAP@50:    {metrics.box.map50:.4f}")
print(f"Overall mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision:         {metrics.box.mp:.4f}")
print(f"Recall:            {metrics.box.mr:.4f}")

Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,353,307 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 103.7±78.3 MB/s, size: 60.0 KB)
val: Scanning /kaggle/input/datasets/smshezanahmed/merged-construction-ppe-dataset/merged_dataset/valid/labels... 2461 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2461/2461 667.9it/s 3.7s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/smshezanahmed/merged-construction-ppe-dataset/merged_dataset/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 154/154 1.9it/s 1:230.5sss
                   all       2461       9294      0.897      0.826      0.898      0.658
                helmet       1692       2982      0.936      0.888      0.927      0.649
                  vest       1484       2503      0.907      0.887       0.93    

## Step 7 — Save results

In [12]:
# ============================================================
# 8. ZIP AND SAVE
# ============================================================
# Zip the new v2 results folder
!zip -r /kaggle/working/yolo26m_ppe.zip /kaggle/working/runs/yolo26m_ppe

print("Saved: /kaggle/working/yolo26m_ppe.zip")

# Generate the download link (IPython is already built-in on Kaggle)
from IPython.display import FileLink
FileLink('/kaggle/working/yolo26m_ppe.zip')

  adding: kaggle/working/runs/yolo26m_ppe/ (stored 0%)
  adding: kaggle/working/runs/yolo26m_ppe/train_batch2.jpg (deflated 2%)
  adding: kaggle/working/runs/yolo26m_ppe/train_batch26251.jpg (deflated 6%)
  adding: kaggle/working/runs/yolo26m_ppe/val_batch1_pred.jpg (deflated 8%)
  adding: kaggle/working/runs/yolo26m_ppe/train_batch1.jpg (deflated 1%)
  adding: kaggle/working/runs/yolo26m_ppe/val_batch2_pred.jpg (deflated 6%)
  adding: kaggle/working/runs/yolo26m_ppe/train_batch26250.jpg (deflated 3%)
  adding: kaggle/working/runs/yolo26m_ppe/confusion_matrix_normalized.png (deflated 22%)
  adding: kaggle/working/runs/yolo26m_ppe/val_batch2_labels.jpg (deflated 6%)
  adding: kaggle/working/runs/yolo26m_ppe/val_batch1_labels.jpg (deflated 9%)
  adding: kaggle/working/runs/yolo26m_ppe/weights/ (stored 0%)
  adding: kaggle/working/runs/yolo26m_ppe/weights/epoch40.pt (deflated 9%)
  adding: kaggle/working/runs/yolo26m_ppe/weights/best.pt (deflated 8%)
  adding: kaggle/working/runs/yolo26m_

/kaggle/working/yolo26m_ppe.zip

In [1]:
from IPython.display import FileLink
FileLink('/kaggle/working/yolo26m_ppe.zip')

/kaggle/working/yolo26m_ppe.zip

In [18]:
import os, yaml, glob
import pandas as pd
from ultralytics import YOLO

# 1. Load the model
model = YOLO("/kaggle/input/models/smshezanahmed/yolo26-ppe-42mb/pytorch/default/1/best (3).pt")
yaml_file = "/kaggle/working/ppe_data/data.yaml"

# ============================================================
# 2. AUTO-FIND DATASET AND FIX YAML
# ============================================================
print("Searching for your dataset images...")
# Search for the valid/images folder anywhere in the Kaggle environment
image_paths = glob.glob('/kaggle/**/valid/images', recursive=True)

if not image_paths:
    print("🚨 ERROR: Could not find your dataset images! 🚨")
    print("Did you forget to add or unzip your PPE dataset into this Kaggle session?")
else:
    # Get the root folder of wherever the dataset actually is
    real_dataset_root = image_paths[0].replace('/valid/images', '').replace('\\', '/')
    print(f"✅ Found dataset at: {real_dataset_root}")
    
    # Read the yaml
    with open(yaml_file, 'r') as f:
        yaml_data = yaml.safe_load(f)

    # Force the paths to match the real location (updated to 'valid')
    yaml_data['path'] = real_dataset_root
    yaml_data['train'] = 'train/images'
    yaml_data['val'] = 'valid/images'  # <--- CHANGED FROM 'val' TO 'valid'
    if 'test' in yaml_data:
        yaml_data['test'] = 'test/images'

    # Save the corrected yaml back
    with open(yaml_file, 'w') as f:
        yaml.dump(yaml_data, f, sort_keys=False)
        
    print("✅ Successfully updated data.yaml with the real path!")

# ============================================================
# 3. RE-RUN VALIDATION (Only if images were found)
# ============================================================
if image_paths:
    metrics = model.val(
        data=yaml_file,
        imgsz=640,
        batch=48,
        device=[0,1],
        plots=True,
        cache = True,
        save_json=True,
        project="/kaggle/working/recovery",
        name="yolo26m_baseline",

    )

    # 4. Print and save all metrics
    CLASS_NAMES = ["helmet", "vest", "person", "no-helmet", "no-vest"]
    rows = []
    for cls_idx, ap50, ap in zip(
        metrics.box.ap_class_index,
        metrics.box.ap50,
        metrics.box.ap
    ):
        name = CLASS_NAMES[cls_idx]
        rows.append({
            "Class": name,
            "AP@50": round(float(ap50), 4),
            "AP@50-95": round(float(ap), 4)
        })

    df = pd.DataFrame(rows)
    print(df.to_markdown(index=False))
    print(f"\nOverall mAP@50:    {metrics.box.map50:.4f}")
    print(f"Overall mAP@50-95: {metrics.box.map:.4f}")
    print(f"Precision:         {metrics.box.mp:.4f}")
    print(f"Recall:            {metrics.box.mr:.4f}")

    df.to_csv("/kaggle/working/recovery/yolo26m_baseline_metrics.csv", index=False)
    print("\nSaved metrics to CSV.")

Searching for your dataset images...
✅ Found dataset at: /kaggle/input/datasets/smshezanahmed/merged-construction-ppe-dataset/merged_dataset
✅ Successfully updated data.yaml with the real path!
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,353,307 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 103.2±30.3 MB/s, size: 53.1 KB)
val: Scanning /kaggle/input/datasets/smshezanahmed/merged-construction-ppe-dataset/merged_dataset/valid/labels... 2461 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2461/2461 1.2Kit/s 2.1s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/smshezanahmed/merged-construction-ppe-dataset/merged_dataset/valid is not writable, cache not saved.
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic altern

In [19]:
# Zip the new v2 results folder
!zip -r /kaggle/working/recovery/yolo26m_baseline9.zip /kaggle/working/recovery/yolo26m_baseline9

print("Saved: /kaggle/working/recovery/yolo26m_baseline9.zip")

# Generate the download link (IPython is already built-in on Kaggle)
from IPython.display import FileLink
FileLink('/kaggle/working/recovery/yolo26m_baseline9.zip')

  adding: kaggle/working/recovery/yolo26m_baseline9/ (stored 0%)
  adding: kaggle/working/recovery/yolo26m_baseline9/val_batch0_pred.jpg (deflated 13%)
  adding: kaggle/working/recovery/yolo26m_baseline9/val_batch2_labels.jpg (deflated 6%)
  adding: kaggle/working/recovery/yolo26m_baseline9/BoxP_curve.png (deflated 10%)
  adding: kaggle/working/recovery/yolo26m_baseline9/BoxR_curve.png (deflated 9%)
  adding: kaggle/working/recovery/yolo26m_baseline9/confusion_matrix_normalized.png (deflated 22%)
  adding: kaggle/working/recovery/yolo26m_baseline9/val_batch2_pred.jpg (deflated 6%)
  adding: kaggle/working/recovery/yolo26m_baseline9/BoxPR_curve.png (deflated 11%)
  adding: kaggle/working/recovery/yolo26m_baseline9/confusion_matrix.png (deflated 22%)
  adding: kaggle/working/recovery/yolo26m_baseline9/val_batch1_pred.jpg (deflated 8%)
  adding: kaggle/working/recovery/yolo26m_baseline9/val_batch1_labels.jpg (deflated 9%)
  adding: kaggle/working/recovery/yolo26m_baseline9/BoxF1_curve.png

/kaggle/working/recovery/yolo26m_baseline9.zip